# Topic 1 · Regression and uncertainty

**What does injection do to the seismicity of a geothermal field, what is that field's
b-value — and how well do you know either answer?**

The two papers we follow:

- **Eberhart-Phillips & Oppenheimer (1984)**, *JGR* 89(B2), `10.1029/JB089iB02p01191` — the first
  decade of this record, analysed, finding *no consistent correlation* between injection and
  seismicity. Part B runs the modern version of that regression.
- **Aki (1965)**, *Bulletin of the Earthquake Research Institute* — the maximum-likelihood estimate of *b* and its
  confidence limits. Part C derives it from the recipe of A.3 and then asks what its interval covers.

The session is in three parts.

**Part A — Linear regression.** Synthetic data generated from known coefficients, so that every
estimate and every interval can be compared against the values that produced them.

**Part B — Injection and seismicity.** The monthly earthquake rate at The Geysers regressed on
water injection, and separately on steam production, with the uncertainty on each slope.

**Part C — The Gutenberg–Richter *b*-value.** Magnitudes above the completeness threshold are
exponentially distributed, so *b* is estimated by maximum likelihood rather than by least
squares, and its standard error follows from the same derivation.


## Part A · Regression and uncertainty, where the answer is known

Everything in this part is generated. We fix a set of coefficients, draw noisy observations from
them, and then estimate the coefficients back — so at every step the estimate can be compared with
the number that produced the data, and every claimed uncertainty can be tested against how often it
is actually right.

The generator is not arbitrary. It is the functional form of a local-magnitude attenuation
relation — amplitude falling with magnitude and distance — with published coefficients used as its
truth, so the numbers are of a size a seismologist will recognise. Nothing about the Earth is being
measured; the point is that the answer is known in advance.


In [ ]:
import importlib.util
import subprocess
import sys
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REQUIRED = ["numpy", "pandas", "matplotlib", "scipy", "requests"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# The same three modules the project notebook uses, so every week reads its data the same way
# and downloads it once. On DataHub they are already beside the notebook; Colab opens the
# notebook alone, so they are fetched -- every run, because a Colab runtime outlives its session
# and a stale copy would silently be used against this week's notebook.
BASE = ("https://raw.githubusercontent.com/AI4EPS/"
        "EPS207_Observational_Seismology/main/docs/notebooks")
if "google.colab" in sys.modules or importlib.util.find_spec("geysers_data") is None:
    for module in ("geysers_data.py", "geysers_func.py", "geysers_plot.py"):
        for attempt in range(4):             # eight people fetch these in the same two minutes
            try:
                urllib.request.urlretrieve(f"{BASE}/{module}", module)
                break
            except Exception as exc:
                if attempt == 3:
                    raise RuntimeError(f"could not fetch {module}: {exc}") from None
                time.sleep(2 * (attempt + 1))
    for name in ("geysers_data", "geysers_func", "geysers_plot"):
        sys.modules.pop(name, None)

from geysers_data import catalog, production, FIELD   # noqa: E402

rng = np.random.default_rng(207)
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True})
print("seed 207 — every number below is reproducible")
print(f"loaders: catalog(), production();  field window {FIELD}")


### A.1 · A model is a design matrix

A seismic station records a peak amplitude $A$ from an earthquake of magnitude $M$ at hypocentral
distance $R$. The relation seismologists use is

$$\log_{10} A \;=\; c_0 \;+\; c_1 M \;+\; c_2 \log_{10} R \;+\; c_3 R \;+\; \varepsilon .$$

The two distance terms carry different physics: $c_2\log_{10}R$ is geometrical spreading, which
falls off as a power of distance, and $c_3 R$ is anelastic attenuation, which is exponential in
distance and so linear in the log. Both are *non-linear functions of $R$* — and neither of them
makes the model non-linear, because what matters is that the relation is **linear in the
parameters** $c_0 \ldots c_3$. Stack one row per reading:

$$y = X\beta + \varepsilon, \qquad
X = \begin{bmatrix} 1 & M_1 & \log_{10}R_1 & R_1 \\ \vdots & \vdots & \vdots & \vdots \end{bmatrix}$$

That matrix $X$ is the **design matrix**, and the columns are whatever functions of the measured
variables the physics asks for. Recognising that a scientific relation is linear in its parameters
is the transferable move: once it is written this way, every linear model in this course is the same
solve, and the choice of columns is where the seismology lives.


In [ ]:
# The truth. These are Hutton & Boore's (1987) southern-California coefficients,
# rewritten from their centred form into the uncentred columns above. Here they are a
# definition, chosen so the synthetic data are of a realistic size.
BETA_TRUE = np.array([-0.591, 1.000, -1.110, -0.00189])   # c0, c1, c2, c3
SIGMA_TRUE = 0.30                                          # log10 units of scatter per reading
NAMES = ["c0 (const)", "c1 (M)", "c2 (log10 R)", "c3 (R)"]

for nm, b in zip(NAMES, BETA_TRUE):
    print(f"{nm:>14s} = {b:9.5f}")
print(f"{'sigma':>14s} = {SIGMA_TRUE:9.5f}")


In [ ]:
# One synthetic network: 2000 readings, magnitudes 1.0-4.5, distances 5-300 km
# drawn log-uniformly so the near field is not swamped by the far field.
n = 2000
M = rng.uniform(1.0, 4.5, n)
R = 10 ** rng.uniform(np.log10(5), np.log10(300), n)

X = np.column_stack([np.ones(n), M, np.log10(R), R])
y = X @ BETA_TRUE + rng.normal(0, SIGMA_TRUE, n)

print("X shape", X.shape, " -> one row per reading, one column per parameter")
print("first three rows of X:")
print(np.round(X[:3], 4))


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
s = ax.scatter(R, y, c=M, s=6, cmap="viridis", alpha=0.6, linewidths=0)
rr = np.logspace(np.log10(5), np.log10(300), 200)
for m in (1.5, 2.5, 3.5, 4.5):
    xx = np.column_stack([np.ones_like(rr), np.full_like(rr, m), np.log10(rr), rr])
    ax.plot(rr, xx @ BETA_TRUE, "k-", lw=1.0)
    ax.text(6, xx[0] @ BETA_TRUE + 0.06, f"M {m}", fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("hypocentral distance R (km)")
ax.set_ylabel(r"$\log_{10} A$")
ax.set_title(f"{n} synthetic readings; black lines are the generator, not a fit")
fig.colorbar(s, ax=ax, label="magnitude")
fig.tight_layout()
plt.show()


### A.2 · Least squares and the normal equations

Choose $\hat\beta$ to minimise the sum of squared residuals,
$S(\beta) = \lVert y - X\beta \rVert^2$.

1. Expand: $S(\beta) = y^\top y - 2\beta^\top X^\top y + \beta^\top X^\top X \beta$.
2. Differentiate with respect to $\beta$:
   $\partial S/\partial\beta = -2X^\top y + 2X^\top X\beta$.
3. Set it to zero. This gives the **normal equations**, $X^\top X\,\hat\beta = X^\top y$, and where
   $X^\top X$ is invertible, $\hat\beta = (X^\top X)^{-1}X^\top y$.
4. The second derivative is $2X^\top X$, which is positive semi-definite for any $X$, so the
   stationary point is a minimum and not a maximum.

Step 3 has a geometric reading worth carrying. Rearranged, it says
$X^\top(y - X\hat\beta) = 0$: the residual vector is **orthogonal to every column of $X$**. Least
squares is the projection of $y$ onto the column space of $X$, and what is left over is, by
construction, the part of the data the model cannot represent. This is why adding a column can only
reduce the residual — and why a reduced residual is not by itself evidence of anything.


In [ ]:
# Two routes to the same estimate. The second is the one to use: lstsq goes through
# a QR/SVD factorisation and does not form X'X, which squares the condition number.
XtX, Xty = X.T @ X, X.T @ y
beta_normal = np.linalg.solve(XtX, Xty)
beta_hat, *_ = np.linalg.lstsq(X, y, rcond=None)

print(f"{'':>14s} {'normal eqns':>12s} {'lstsq':>12s} {'truth':>12s}")
for nm, a, b, t in zip(NAMES, beta_normal, beta_hat, BETA_TRUE):
    print(f"{nm:>14s} {a:12.5f} {b:12.5f} {t:12.5f}")
print(f"\nlargest disagreement between the two routes: {np.abs(beta_normal - beta_hat).max():.2e}")


In [ ]:
# The orthogonality of step 3, checked rather than asserted.
resid = y - X @ beta_hat
print("X' r, one entry per column of X:", np.array2string(X.T @ resid, precision=9))
print(f"largest |X' r| = {np.abs(X.T @ resid).max():.2e}  (zero, to machine precision)")

rss = resid @ resid
p = X.shape[1]
sigma_hat = np.sqrt(rss / (n - p))
print(f"\nsigma_hat = sqrt(RSS/(n-p)) = {sigma_hat:.4f}   against a true {SIGMA_TRUE}")


### A.3 · Least squares is maximum likelihood under Gaussian noise

Nothing so far said anything about probability: we minimised a sum of squares because it was a
convenient thing to minimise. Now assume the errors are independent and Gaussian,
$\varepsilon_i \sim N(0,\sigma^2)$, and the choice stops being a convention.

1. The density of one observation is
   $p(y_i \mid \beta,\sigma) = (2\pi\sigma^2)^{-1/2}\exp\!\big[-(y_i - x_i^\top\beta)^2/2\sigma^2\big]$.
2. Independence makes the likelihood the product,
   $L(\beta,\sigma) = \prod_i p(y_i\mid\beta,\sigma)$.
3. Take the log, which is monotone and so does not move the maximum:
   $\ell(\beta,\sigma) = -\tfrac{n}{2}\log(2\pi\sigma^2) - \dfrac{1}{2\sigma^2}\lVert y - X\beta\rVert^2$.
4. Only the last term contains $\beta$, and it is $-S(\beta)/2\sigma^2$ — a negative constant times
   the sum of squares. **Maximising $\ell$ over $\beta$ is therefore exactly minimising
   $S(\beta)$**, whatever $\sigma$ happens to be.

So least squares is not a definition; it is the maximum-likelihood estimator *for one particular
noise model*. That distinction is the whole of Part C.

It also gives the recipe this session uses three times:

> **(i)** write down the likelihood of the data under the model;
> **(ii)** maximise it to get the estimate;
> **(iii)** take the curvature of $\ell$ at the maximum to get the uncertainty.

Steps (i) and (iii) never mention Gaussians. Only step (ii) collapses into the normal equations, and
only when the noise is Gaussian.


In [ ]:
def loglik(b, sig):
    r = y - X @ b
    return -0.5 * n * np.log(2 * np.pi * sig**2) - (r @ r) / (2 * sig**2)

# Slice the log-likelihood along c1 with the other three held at their fitted values,
# and overlay the sum of squares rescaled by -1/(2 sigma^2). Step 4 says they coincide.
c1_grid = np.linspace(beta_hat[1] - 0.05, beta_hat[1] + 0.05, 201)
ll = np.array([loglik(np.r_[beta_hat[0], c, beta_hat[2:]], sigma_hat) for c in c1_grid])
sse = np.array([((y - X @ np.r_[beta_hat[0], c, beta_hat[2:]])**2).sum() for c in c1_grid])

print(f"c1 maximising the log-likelihood: {c1_grid[ll.argmax()]:.5f}")
print(f"c1 minimising the sum of squares: {c1_grid[sse.argmin()]:.5f}")
print(f"c1 from lstsq:                    {beta_hat[1]:.5f}")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.4))
a1.plot(c1_grid, ll, lw=1.6, label="log-likelihood")
a1.plot(c1_grid, -sse / (2 * sigma_hat**2) + (ll.max() + sse.min() / (2 * sigma_hat**2)),
        "--", lw=1.6, label=r"$-S(\beta)/2\sigma^2$ + const")
a1.axvline(beta_hat[1], color="k", lw=0.8)
a1.set_xlabel(r"$c_1$"); a1.set_ylabel("value"); a1.legend(fontsize=7)
a1.set_title("same maximiser, two derivations")

g2, g3 = np.meshgrid(np.linspace(-1.35, -0.90, 90), np.linspace(-0.0035, -0.0005, 90))
Z = np.array([loglik(np.r_[beta_hat[:2], u, v], sigma_hat)
              for u, v in zip(g2.ravel(), g3.ravel())]).reshape(g2.shape)
a2.contour(g2, g3, Z, levels=np.percentile(Z, [88, 94, 97, 99, 99.7]), linewidths=0.8)
a2.plot(*beta_hat[2:], "r+", ms=10)
a2.set_xlabel(r"$c_2$"); a2.set_ylabel(r"$c_3$")
a2.set_title("log-likelihood in the two distance terms")
fig.tight_layout(); plt.show()


### A.4 · Where the error bars come from

$\hat\beta$ is a function of noisy data, so it is itself a random variable. Substitute
$y = X\beta + \varepsilon$ into the estimator:

$$\hat\beta = (X^\top X)^{-1}X^\top(X\beta + \varepsilon) = \beta + (X^\top X)^{-1}X^\top\varepsilon .$$

1. The second term has mean zero, so $\hat\beta$ is **unbiased**: on average it is the truth.
2. Its covariance is
   $\operatorname{Cov}(\hat\beta) = (X^\top X)^{-1}X^\top \operatorname{Cov}(\varepsilon) X (X^\top X)^{-1}$,
   and with $\operatorname{Cov}(\varepsilon) = \sigma^2 I$ the middle collapses and this is
   $\boxed{\sigma^2 (X^\top X)^{-1}}$.
3. $\sigma$ is unknown, so use $\hat\sigma^2 = \mathrm{RSS}/(n-p)$; the divisor is $n-p$ rather than
   $n$ because $p$ directions of the residual were spent fitting.

Now connect it to step (iii) of the recipe. Differentiate $\ell$ from A.3 twice:
$-\partial^2\ell/\partial\beta\partial\beta^\top = X^\top X/\sigma^2$. That is the **Fisher
information** — how sharply peaked the likelihood is — and $\operatorname{Cov}(\hat\beta)$ is its
inverse. A sharp peak is a well-determined parameter.

This is the sentence that survives into Part C: *the uncertainty is the inverse curvature of the
log-likelihood at its maximum.* The Gaussian case is the one where that curvature happens to have
the closed form $\sigma^2(X^\top X)^{-1}$.


In [ ]:
Cov = sigma_hat**2 * np.linalg.inv(XtX)
se = np.sqrt(np.diag(Cov))

print(f"{'':>14s} {'estimate':>11s} {'std error':>11s} {'truth':>11s} {'(est-truth)/se':>15s}")
for nm, b, s_, t in zip(NAMES, beta_hat, se, BETA_TRUE):
    print(f"{nm:>14s} {b:11.5f} {s_:11.5f} {t:11.5f} {(b - t) / s_:15.2f}")

fisher = XtX / sigma_hat**2
print(f"\nlargest |Cov - inverse Fisher information| = "
      f"{np.abs(Cov - np.linalg.inv(fisher)).max():.2e}")


**Exercise 1.** A standard error is a claim about repetition: over many repeats of the
experiment, the interval $\hat\beta_j \pm 1.96\,\mathrm{se}_j$ should contain the true $c_j$ about
95 % of the time. That claim is testable here and nowhere in Parts B or C, because only here is the
truth known.

Regenerate `y` from `BETA_TRUE` 1000 times using the same design matrix `X`, refit each time, and
report — for each coefficient — the fraction of refits whose interval covered the truth.


In [ ]:
# your code here


In [ ]:
# ── Checkpoint 1 ── run this if you are behind or something broke ──
# State Part A needs from here on: the design, the fit, and its covariance.
n, p = 2000, 4
BETA_TRUE = np.array([-0.591, 1.000, -1.110, -0.00189]); SIGMA_TRUE = 0.30
_r = np.random.default_rng(207)
M = _r.uniform(1.0, 4.5, n); R = 10 ** _r.uniform(np.log10(5), np.log10(300), n)
X = np.column_stack([np.ones(n), M, np.log10(R), R])
y = X @ BETA_TRUE + _r.normal(0, SIGMA_TRUE, n)
XtX = X.T @ X
beta_hat, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta_hat
sigma_hat = np.sqrt(resid @ resid / (n - p))
Cov = sigma_hat**2 * np.linalg.inv(XtX)
se = np.sqrt(np.diag(Cov))
print("recovered:", np.round(beta_hat, 4))


### A.5 · Confidence interval versus prediction interval

Two questions get asked of a fitted curve, and they are not the same question.

**Where does the curve go?** The fitted value at a new row $x_0$ is $x_0^\top\hat\beta$, a linear
function of $\hat\beta$, so from A.4

$$\operatorname{Var}(x_0^\top\hat\beta) = x_0^\top \operatorname{Cov}(\hat\beta)\, x_0
= \sigma^2\, x_0^\top (X^\top X)^{-1} x_0 .$$

Every term carries a factor $\sigma^2/n$, so this **shrinks as $1/\sqrt n$** and goes to zero with
enough data. It is the uncertainty in the *mean* relation.

**What will the next station read?** A new observation is $y_0 = x_0^\top\beta + \varepsilon_0$, and
$\varepsilon_0$ is a fresh draw, independent of everything used to fit. So

$$\operatorname{Var}(y_0 - x_0^\top\hat\beta)
= \underbrace{\sigma^2 x_0^\top (X^\top X)^{-1} x_0}_{\text{where the curve is}}
+ \underbrace{\sigma^2}_{\text{scatter of one reading}} .$$

The second term does not contain $n$. **No amount of data shrinks it**, because it is not ignorance
about the model — it is the spread of the thing being predicted.

Confusing the two is the commonest error in applied regression, and it is not a subtlety: below,
the two intervals differ by more than an order of magnitude on the same fit at the same point.


In [ ]:
x0 = np.array([1.0, 3.0, np.log10(50.0), 50.0])      # M 3.0 at R 50 km
var_mean = x0 @ Cov @ x0

ci = 1.96 * np.sqrt(var_mean)
pi = 1.96 * np.sqrt(var_mean + sigma_hat**2)

print(f"at M 3.0, R 50 km, fitted log10 A = {x0 @ beta_hat:.4f}")
print(f"  95% confidence interval on the curve      +/- {ci:.4f}")
print(f"  95% prediction interval for one reading   +/- {pi:.4f}")
print(f"  ratio {pi / ci:.1f}x")


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
rr = np.logspace(np.log10(5), np.log10(300), 300)
Xs = np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr])
mu = Xs @ beta_hat
vm = np.einsum("ij,jk,ik->i", Xs, Cov, Xs)

keep = np.abs(M - 3.0) < 0.25
ax.scatter(R[keep], y[keep], s=8, color="0.55", label="readings, M 3.0 +/- 0.25")
ax.fill_between(rr, mu - 1.96 * np.sqrt(vm + sigma_hat**2), mu + 1.96 * np.sqrt(vm + sigma_hat**2),
                color="C1", alpha=0.22, label="95% prediction interval")
ax.fill_between(rr, mu - 1.96 * np.sqrt(vm), mu + 1.96 * np.sqrt(vm),
                color="C0", alpha=0.85, label="95% confidence interval")
ax.plot(rr, mu, "k-", lw=1.2, label="fitted curve at M 3.0")
ax.set_xscale("log"); ax.set_xlabel("R (km)"); ax.set_ylabel(r"$\log_{10} A$")
ax.set_title("the same fit, two intervals"); ax.legend(fontsize=7, loc="lower left")
fig.tight_layout(); plt.show()


In [ ]:
# How each interval behaves as the network grows.
sizes = np.array([25, 50, 100, 250, 500, 1000, 2000, 5000, 10000])
rows = []
for m_ in sizes:
    r_ = np.random.default_rng(9000 + m_)
    Mm = r_.uniform(1.0, 4.5, m_); Rm = 10 ** r_.uniform(np.log10(5), np.log10(300), m_)
    Xm = np.column_stack([np.ones(m_), Mm, np.log10(Rm), Rm])
    ym = Xm @ BETA_TRUE + r_.normal(0, SIGMA_TRUE, m_)
    bm, *_ = np.linalg.lstsq(Xm, ym, rcond=None)
    rm = ym - Xm @ bm; sm2 = rm @ rm / (m_ - p)
    vm_ = x0 @ (sm2 * np.linalg.inv(Xm.T @ Xm)) @ x0
    rows.append((m_, 1.96 * np.sqrt(vm_), 1.96 * np.sqrt(vm_ + sm2)))

print(f"{'n':>7s} {'CI half-width':>15s} {'PI half-width':>15s}")
for m_, c_, pp in rows:
    print(f"{m_:7d} {c_:15.4f} {pp:15.4f}")


In [ ]:
arr = np.array(rows)
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.loglog(arr[:, 0], arr[:, 1], "o-", label="confidence interval")
ax.loglog(arr[:, 0], arr[:, 2], "s-", label="prediction interval")
ax.loglog(arr[:, 0], arr[0, 1] * np.sqrt(arr[0, 0] / arr[:, 0]), "k--", lw=0.8,
          label=r"$1/\sqrt{n}$")
ax.axhline(1.96 * SIGMA_TRUE, color="0.5", lw=0.8, ls=":")
ax.text(30, 1.96 * SIGMA_TRUE * 1.06, r"1.96 $\sigma$ — the floor", fontsize=7, color="0.4")
ax.set_xlabel("number of readings"); ax.set_ylabel("half-width at M 3.0, R 50 km")
ax.legend(fontsize=7); ax.set_title("one shrinks, one does not")
fig.tight_layout(); plt.show()


**Exercise 2.** The previous exercise showed the confidence interval covers *the truth*
about 95 % of the time. Now ask what it does to the question people actually use it for.

Over 2000 trials, draw a fresh reading $y_0$ at $x_0$ (M 3.0, R 50 km), refit on new data, and
count how often that reading falls inside the confidence band and how often inside the prediction
band. One of the two answers is 95 %. Predict which, and by roughly how much the other misses,
before you run it.


In [ ]:
# your code here


In [ ]:
# ── Checkpoint 2 ── run this if you are behind or something broke ──
# Nothing after A.5 depends on the exercises; this restates the two intervals at x0.
x0 = np.array([1.0, 3.0, np.log10(50.0), 50.0])
var_mean = x0 @ Cov @ x0
print(f"CI +/- {1.96*np.sqrt(var_mean):.4f}   PI +/- {1.96*np.sqrt(var_mean + sigma_hat**2):.4f}")


### A.6 · Collinearity: a good fit with meaningless coefficients

$\hat\beta$ came back close to the truth and every standard error was small. That is a property of
this design, not of least squares, and the two distance columns are about to show why.

$\log_{10}R$ and $R$ are different functions, but over a bounded range of $R$ they move together.
When two columns of $X$ are nearly proportional, $X^\top X$ is nearly singular, and
$(X^\top X)^{-1}$ — which is the covariance — has large entries in exactly those directions. The
consequence is specific and worth stating precisely:

> The **combination** $c_2\log_{10}R + c_3R$ is determined by the data. The **individual**
> $c_2$ and $c_3$ are not.

A published coefficient can therefore disagree with yours by many standard errors while both curves
pass through the same points, and reporting either coefficient alone, with its own error bar, hides
that the two are being traded against each other.


In [ ]:
corr_cols = np.corrcoef(np.log10(R), R)[0, 1]
D = np.sqrt(np.outer(np.diag(Cov), np.diag(Cov)))
corr_coef = (Cov / D)[2, 3]

print(f"correlation of the two COLUMNS,      log10 R vs R : {corr_cols:+.4f}")
print(f"correlation of the two COEFFICIENTS, c2 vs c3     : {corr_coef:+.4f}")
print(f"condition number of X                             : {np.linalg.cond(X):.1f}")
print("\nsingular values of X:", np.array2string(np.linalg.svd(X, compute_uv=False), precision=3))


In [ ]:
# Walk to the two ends of the long axis of the 95% ellipse for (c2, c3) and ask
# whether the predicted curves can be told apart.
cov_dist = Cov[2:, 2:]
w, V = np.linalg.eigh(cov_dist)
axis = V[:, -1] * np.sqrt(w[-1] * stats.chi2.ppf(0.95, 2))
end_a, end_b = beta_hat[2:] + axis, beta_hat[2:] - axis

curves = []
for end in (end_a, end_b):
    b_alt = np.r_[beta_hat[:2], end]
    curves.append(np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr]) @ b_alt)

print(f"corner A of the ellipse: c2 = {end_a[0]:+.4f}, c3 = {end_a[1]:+.6f}")
print(f"corner B of the ellipse: c2 = {end_b[0]:+.4f}, c3 = {end_b[1]:+.6f}")
gap = np.abs(curves[0] - curves[1]).max()
print(f"c2 differs between the corners by {abs(end_a[0]-end_b[0]) / se[2]:.1f} standard errors,")
print(f"yet the two predicted curves differ by at most {gap:.4f} log10 units -- which is "
      f"{gap / (1.96*np.sqrt(var_mean + sigma_hat**2)):.2f} of the")
print("half-width of the prediction interval for a single reading, from A.5.")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.5))
th = np.linspace(0, 2 * np.pi, 200)
ell = (V @ np.diag(np.sqrt(w * stats.chi2.ppf(0.95, 2))) @ np.vstack([np.cos(th), np.sin(th)])).T
a1.plot(beta_hat[2] + ell[:, 0], beta_hat[3] + ell[:, 1], "C0-", lw=1.2)
a1.plot(*beta_hat[2:], "C0+", ms=9, label="fit")
a1.plot(*BETA_TRUE[2:], "kx", ms=8, label="truth")
a1.plot([end_a[0], end_b[0]], [end_a[1], end_b[1]], "C3o", ms=5, label="ellipse ends")
a1.set_xlabel(r"$c_2$"); a1.set_ylabel(r"$c_3$"); a1.legend(fontsize=7)
a1.set_title("95% joint region: a ridge, not a blob")

mu_ = np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr]) @ beta_hat
vm_ = np.einsum("ij,jk,ik->i", np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0),
                                                np.log10(rr), rr]), Cov,
                np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr]))
a2.scatter(R[np.abs(M - 3.0) < 0.25], y[np.abs(M - 3.0) < 0.25], s=6, color="0.75", zorder=0)
a2.fill_between(rr, mu_ - 1.96 * np.sqrt(vm_ + sigma_hat**2),
                mu_ + 1.96 * np.sqrt(vm_ + sigma_hat**2), color="C1", alpha=0.18, zorder=1,
                label="95% prediction interval")
a2.plot(rr, curves[0], "C3-", lw=1.4, zorder=3, label="corner A")
a2.plot(rr, curves[1], "C3--", lw=1.4, zorder=3, label="corner B")
a2.set_xscale("log"); a2.set_xlabel("R (km)"); a2.set_ylabel(r"$\log_{10} A$ at M 3.0")
a2.legend(fontsize=7, loc="lower left")
a2.set_title("both are consistent with these data")
fig.tight_layout(); plt.show()


It is tempting to look for a reparameterisation that makes the problem go away. Hutton &
Boore write their distance terms centred on a reference distance, $\log_{10}(R/100)$ and $R-100$,
and centring is the standard advice for collinearity. Test it.


In [ ]:
Xc = np.column_stack([np.ones(n), M, np.log10(R / 100.0), R - 100.0])
bc, *_ = np.linalg.lstsq(Xc, y, rcond=None)
rc = y - Xc @ bc
Cc = (rc @ rc / (n - p)) * np.linalg.inv(Xc.T @ Xc)
Dc = np.sqrt(np.outer(np.diag(Cc), np.diag(Cc)))

print(f"uncentred: corr(c2, c3) = {corr_coef:+.4f}, cond(X) = {np.linalg.cond(X):8.1f}")
print(f"centred  : corr(c2, c3) = {(Cc/Dc)[2, 3]:+.4f}, cond(X) = {np.linalg.cond(Xc):8.1f}")
print(f"\nresidual sum of squares, uncentred {resid @ resid:.4f} vs centred {rc @ rc:.4f}"
      "   -- the same fit in different coordinates")


The conditioning improves and the intercept becomes interpretable — it is now the value at
the reference distance rather than an extrapolation to $R=1$ km. **The correlation between the two
slopes does not move at all.** Shifting a column by a constant changes how that column relates to
the intercept; it cannot change how two slopes trade against each other. Centring is worth doing,
and it is not a cure.

What is actually identified is a *direction*. The eigenvectors of the $(c_2,c_3)$ covariance are the
long and short axes of the ellipse: one combination the data pin down, one they do not.


In [ ]:
# Do this on the CORRELATION matrix, not the covariance. c3 is a thousand times smaller
# than c2, so eigenvectors of the raw covariance report the choice of units, not the
# identifiability. Measuring each coefficient in its own standard errors removes that.
corr2 = np.array([[1.0, corr_coef], [corr_coef, 1.0]])
wc, Vc = np.linalg.eigh(corr2)
print(f"well-determined direction: {Vc[0, 0]:+.3f} c2/se2 {Vc[1, 0]:+.3f} c3/se3,"
      f"  sd {np.sqrt(wc[0]):.3f}")
print(f"poorly determined:         {Vc[0, 1]:+.3f} c2/se2 {Vc[1, 1]:+.3f} c3/se3,"
      f"  sd {np.sqrt(wc[1]):.3f}")
print(f"the data constrain one direction {np.sqrt(wc[1] / wc[0]):.1f}x better than the other")
print("\nThe well-determined direction moves the two coefficients TOGETHER; the poorly")
print("determined one trades them against each other, which is what a negative correlation")
print("means. Report the combination, or drop a term -- c2 alone, with its own error bar,")
print("states a precision the data never had.")


In [ ]:
# ── Checkpoint 3 ── run this if you are behind or something broke ──
# Everything A.7 needs.
Cov = sigma_hat**2 * np.linalg.inv(XtX)
se = np.sqrt(np.diag(Cov))
sv = np.linalg.svd(X, compute_uv=False)
print("singular values:", np.round(sv, 3))


### A.7 · Ridge regression is MAP under a Gaussian prior

The trade-off in A.6 is the data declining to separate two directions. One response is to say
something, before looking at the data, about how large the coefficients ought to be. Doing that
carefully turns out to give a familiar formula.

1. **The forward model, probabilistically.** $y = X\beta + \varepsilon$ with
   $\varepsilon\sim N(0,\sigma^2 I)$, so
   $p(y\mid\beta) \propto \exp\!\big[-\lVert y-X\beta\rVert^2/2\sigma^2\big]$ — the A.3 likelihood.
2. **The prior.** State it before the data: $\beta \sim N(0, s^2 I)$, so
   $p(\beta) \propto \exp\!\big[-\lVert\beta\rVert^2/2s^2\big]$. Here $s$ is a real quantity in the
   units of $\beta$ — the size of coefficient we would be unsurprised by.
3. **Bayes.** $p(\beta\mid y) \propto p(y\mid\beta)\,p(\beta)$. The denominator does not contain
   $\beta$, so it cannot move the maximum and is dropped.
4. **Negative log.** The product becomes a sum of two quadratics:
   $-\log p(\beta\mid y) = \dfrac{\lVert y - X\beta\rVert^2}{2\sigma^2} + \dfrac{\lVert\beta\rVert^2}{2s^2} + \text{const}$.
5. **Rescale** by $2\sigma^2$, which is positive and so does not move the minimiser:
   $\lVert y-X\beta\rVert^2 + \lambda\lVert\beta\rVert^2$ with
   $\boxed{\lambda = \sigma^2/s^2}$. This is the damped objective, and $\lambda$ is now *read off*
   rather than tuned: it is the ratio of how noisy the data are to how large the coefficients were
   expected to be.
6. **Minimise.** Differentiating gives $(X^\top X + \lambda I)\hat\beta_\lambda = X^\top y$. Adding
   $\lambda I$ raises every eigenvalue of $X^\top X$ by $\lambda$, so for $\lambda>0$ the matrix is
   positive definite and invertible **even when $X^\top X$ is singular** — which is what makes this
   the standard move for an under-determined inverse problem, in week 12 and again in week 14.
7. **The mechanism.** Write $X = UDV^\top$. Then
   $\hat\beta_\lambda = \sum_i f_i \dfrac{u_i^\top y}{d_i} v_i$ with **filter factors**
   $f_i = d_i^2/(d_i^2+\lambda)$. Directions the data constrain well ($d_i^2 \gg \lambda$) pass
   through untouched; directions the data barely see are damped toward zero. Ridge does not shrink
   $\beta$ uniformly — it shrinks *the directions the data are quiet about*.

In practice the columns are standardised first and the intercept is left unpenalised, so that the
prior means the same thing regardless of the units each column happens to be in.


In [ ]:
# Filter factors on this design, at five values of lambda.
print(f"{'lambda':>10s}  " + "  ".join(f"f{i+1}" for i in range(p)))
for lam in [0.0, 1e0, 1e2, 1e4, 1e6]:
    f = sv**2 / (sv**2 + lam)
    print(f"{lam:10.0e}  " + "  ".join(f"{v:.4f}" for v in f))
print("\nthe best-resolved direction is untouched at every lambda; the weakest is damped first")


Filter factors describe what ridge *does*; they do not say when it is worth doing. On the
four-column design it is not: those four parameters are estimated from 2000 readings, every
singular value is comfortably above zero, and a prior displaces a well-determined fit for nothing.

The prior earns its place when the model has many parameters that each see little data. The
canonical case in this problem is a **site term**: one coefficient per station, absorbing the fact
that a station on soft sediment reads systematically higher than one on rock. Add sixty of them and
the design has sixty-four columns, four readings per station — and each site term is estimated from
those four readings alone.

Now $\lambda$ stops being a knob. Step 5 of the derivation gives $\lambda = \sigma^2/s^2$, and both
quantities are measurable: $\sigma$ is the scatter of repeat readings at one station, $s$ the
spread of the site terms across stations. The values below were measured on a real regional
network.


In [ ]:
SIGMA_W, TAU_SITE, N_STA = 0.239, 0.245, 60   # within-station, between-station, stations
SITE_LAM = SIGMA_W**2 / TAU_SITE**2

def site_design(r_, m_):
    """The A.1 design, plus one indicator column per station."""
    Mm = r_.uniform(1.0, 4.5, m_)
    Rm = 10 ** r_.uniform(np.log10(5), np.log10(300), m_)
    which = r_.integers(0, N_STA, m_)
    D = np.zeros((m_, N_STA)); D[np.arange(m_), which] = 1.0
    return np.column_stack([np.ones(m_), Mm, np.log10(Rm), Rm, D])

def site_ridge(Xm, ym, lam):
    """Penalise the site columns only: the prior is about stations, not about attenuation."""
    pen = np.zeros(Xm.shape[1]); pen[4:] = lam
    return np.linalg.solve(Xm.T @ Xm + np.diag(pen), Xm.T @ ym)

_probe = site_design(np.random.default_rng(0), 250)
_empty = int((_probe[:, 4:].sum(axis=0) == 0).sum())
print(f"design: {_probe.shape[1]} columns, rank {np.linalg.matrix_rank(_probe)}")
print("  -1  the intercept column is the sum of the sixty site columns")
print(f"  -{_empty}  station(s) drew no readings at all, so their column is identically zero")
print("X'X is therefore SINGULAR: only DIFFERENCES between site terms are identifiable, and a")
print("station with no data has no estimate. Step 6 of the derivation is not hypothetical here")
print("-- for lambda > 0 the matrix is invertible, which is the only reason a fit exists.\n")
print(f"within-station scatter  sigma = {SIGMA_W}")
print(f"between-station scatter s     = {TAU_SITE}")
print(f"so the derivation predicts lambda = sigma^2 / s^2 = {SITE_LAM:.2f}")


In [ ]:
LAMS = np.logspace(-2, 3, 26)

def held_out(m_, trials=200, ntest=800):
    """Mean squared error on fresh readings from the same stations."""
    err = np.zeros(len(LAMS))
    for t in range(trials):
        r_ = np.random.default_rng(7000 + t)
        site = r_.normal(0, TAU_SITE, N_STA)
        full = np.r_[BETA_TRUE, site]
        Xtr = site_design(r_, m_); ytr = Xtr @ full + r_.normal(0, SIGMA_W, m_)
        Xte = site_design(r_, ntest); yte = Xte @ full + r_.normal(0, SIGMA_W, ntest)
        for i, lam in enumerate(LAMS):
            err[i] += np.mean((yte - Xte @ site_ridge(Xtr, ytr, lam)) ** 2)
    return err / trials

curves_site = {m_: held_out(m_) for m_ in (250, 600, 3000)}
for m_, c in curves_site.items():
    j = int(c.argmin())
    print(f"n = {m_:5d} ({m_ / N_STA:4.1f} readings/station): best lambda {LAMS[j]:6.3f}, "
          f"held-out error {c[j]:.4f} against {c[0]:.4f} at the weakest prior tried "
          f"({100 * (c[0] - c[j]) / c[0]:+.1f}%)")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.4))
for m_, c in curves_site.items():
    a1.semilogx(LAMS, c, "o-", ms=3, label=f"n = {m_}")
    a1.plot(LAMS[c.argmin()], c.min(), "k*", ms=11)
a1.axvline(SITE_LAM, color="C3", ls="--", lw=1.0)
a1.text(SITE_LAM * 1.15, a1.get_ylim()[1] * 0.97, r"$\sigma^2/s^2$", color="C3", fontsize=8,
        va="top")
a1.set_xlabel(r"$\lambda$"); a1.set_ylabel("held-out mean squared error")
a1.legend(fontsize=7); a1.set_title("the prior pays when each station is thin")

_r = np.random.default_rng(7000)
site_true = _r.normal(0, TAU_SITE, N_STA)
Xs_ = site_design(_r, 250); ys_ = Xs_ @ np.r_[BETA_TRUE, site_true] + _r.normal(0, SIGMA_W, 250)
for lam, col in ((1e-2, "0.6"), (SITE_LAM, "C0")):
    est = site_ridge(Xs_, ys_, lam)[4:]
    a2.plot(site_true - site_true.mean(), est - est.mean(), "o", ms=4, color=col,
            label=f"lambda = {lam:.2f}")
a2.plot([-0.7, 0.7], [-0.7, 0.7], "k-", lw=0.8)
a2.set_xlabel("true site term (centred)"); a2.set_ylabel("estimated site term (centred)")
a2.legend(fontsize=7); a2.set_title("shrinkage, station by station")
fig.tight_layout(); plt.show()


**Exercise 3.** The curve on the left cannot be drawn without knowing the true site terms,
so it is not available on real data. Cross-validation is: hold out part of the data, fit on the
rest, score the prediction on what was held out — and it never looks at the truth.

Build one 250-reading network, run 5-fold cross-validation over `LAMS`, and report the $\lambda$ it
selects. Compare it with $\sigma^2/s^2$ from the derivation. Agreement here is the claim of step 5
being confirmed by a procedure that knows nothing about it.


In [ ]:
# your code here


### A.8 · When the noise is not Gaussian

Return to the recipe in A.3: **(i)** write the likelihood, **(ii)** maximise it, **(iii)** invert
the curvature at the maximum. Steps (i) and (iii) are general. Step (ii) collapsed into the normal
equations only because the log-likelihood was a sum of squares — and it was a sum of squares only
because the noise was Gaussian.

Take a quantity $x \ge x_{\min}$ that is exponentially distributed,
$p(x) = \beta e^{-\beta(x - x_{\min})}$. Two ways to estimate $\beta$ from a sample:

**Least squares on a histogram.** Bin the sample, count how many observations exceed each bin edge,
take $\log_{10}$ of the counts, and fit a straight line. The slope gives $\beta$. This is a
regression, so all of A.2–A.5 applies — except that its assumptions do not: the counts are Poisson,
so their scatter grows with the count rather than being constant; and each cumulative count contains
every count above it, so the points are strongly correlated rather than independent. Both violations
are invisible in the plot, which looks like an excellent straight line.

**Maximum likelihood.** Follow the recipe. The log-likelihood is
$\ell(\beta) = n\log\beta - \beta\sum_i (x_i - x_{\min})$; setting $\partial\ell/\partial\beta = 0$
gives $\hat\beta = 1/(\bar x - x_{\min})$, and $-\partial^2\ell/\partial\beta^2 = n/\beta^2$, so by
step (iii) the standard error is $\hat\beta/\sqrt{n}$. No design matrix, no regression, and an error
bar that came from the same place as the one in A.4.

Both are computed below on the same samples, and — because the data are generated — both can be
judged against the truth and against how much they actually vary from sample to sample.


In [ ]:
BETA_EXP = np.log(10)      # rate of the exponential; the truth to recover
X_MIN, BIN = 0.0, 0.1      # values reported rounded to the nearest 0.1
N_SAMPLE = 1000

def one_sample(seed):
    r_ = np.random.default_rng(seed)
    x = (X_MIN - BIN / 2) + r_.exponential(1 / BETA_EXP, N_SAMPLE)
    xb = np.round(x / BIN) * BIN                        # what the catalogue would report
    mle = 1.0 / (xb.mean() - X_MIN + BIN / 2)           # rounding shifts the effective minimum
    edges = np.round(np.arange(X_MIN, xb.max() + BIN, BIN), 4)
    cnt = np.array([(xb >= e - 1e-9).sum() for e in edges])
    k = cnt > 0
    A = np.column_stack([np.ones(k.sum()), edges[k]])
    g, *_ = np.linalg.lstsq(A, np.log10(cnt[k]), rcond=None)
    rr_ = np.log10(cnt[k]) - A @ g
    se_ls = np.sqrt(np.diag((rr_ @ rr_ / (k.sum() - 2)) * np.linalg.inv(A.T @ A)))[1]
    return mle, mle / np.sqrt(N_SAMPLE), -g[1] * np.log(10), se_ls * np.log(10)

print("one sample:", np.round(one_sample(0), 4))


In [ ]:
draws = np.array([one_sample(s) for s in range(400)])
mle, mle_se, lsq, lsq_se = draws.T

print(f"truth                                       {BETA_EXP:8.4f}")
print(f"maximum likelihood, mean over 400 samples   {mle.mean():8.4f}")
print(f"least squares,      mean over 400 samples   {lsq.mean():8.4f}"
      f"   ({100 * (lsq.mean() - BETA_EXP) / BETA_EXP:+.1f}% biased)")
print()
print(f"actual spread of the MLE           {mle.std():8.4f}   it reports {mle_se.mean():8.4f}")
print(f"actual spread of least squares     {lsq.std():8.4f}   it reports {lsq_se.mean():8.4f}")
print()
print(f"least squares is {lsq.std() / mle.std():.1f}x more variable than the MLE,")
print(f"and understates its own spread by a factor of {lsq.std() / lsq_se.mean():.1f}")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.4))
r_ = np.random.default_rng(4)
x = (X_MIN - BIN / 2) + r_.exponential(1 / BETA_EXP, N_SAMPLE)
xb = np.round(x / BIN) * BIN
edges = np.round(np.arange(X_MIN, xb.max() + BIN, BIN), 4)
cnt = np.array([(xb >= e - 1e-9).sum() for e in edges])
k = cnt > 0
a1.semilogy(edges[k], cnt[k], "o", ms=3.5, color="0.3")
gg, *_ = np.linalg.lstsq(np.column_stack([np.ones(k.sum()), edges[k]]), np.log10(cnt[k]), rcond=None)
a1.semilogy(edges[k], 10 ** (gg[0] + gg[1] * edges[k]), "C3-", lw=1.2, label="least squares")
a1.set_xlabel("x"); a1.set_ylabel("count of values above x")
a1.set_title("the fit that looks unimpeachable"); a1.legend(fontsize=7)

bins = np.linspace(min(lsq.min(), mle.min()), max(lsq.max(), mle.max()), 45)
a2.hist(lsq, bins=bins, alpha=0.6, label="least squares", color="C3")
a2.hist(mle, bins=bins, alpha=0.6, label="maximum likelihood", color="C0")
a2.axvline(BETA_EXP, color="k", lw=1.2, label="truth")
a2.set_xlabel(r"estimated $\beta$"); a2.set_ylabel("samples")
a2.set_title("400 samples, same data, two estimators"); a2.legend(fontsize=7)
fig.tight_layout(); plt.show()


The straight line on the left is not a bad fit — it is a very good fit, to the wrong
objective. Least squares here is still minimising a sum of squares; it is just no longer the
maximum-likelihood estimator, because the noise it implicitly assumes is not the noise the data
have. The price is paid three times over: the estimate is biased, it varies more from sample to
sample than it needs to, and the standard error it reports is the one belonging to the model it
assumed rather than the one that generated the data.

The third of those is the dangerous one. A biased estimate with an honest error bar advertises its
own trouble. This one does not.

Part C is this section with $x$ replaced by earthquake magnitude — where the same argument
turns out to bite in a different place.


## Part B · Does injection drive the seismicity?

The Geysers is the largest geothermal field in the world. Water is pumped into it every month and
steam is drawn out of it every month, and the question the field is known for is whether the
pumping is what makes it shake.

Section 10 of the project notebook answers it with a correlation. This part asks the same question
as a regression, because a regression gives what a correlation cannot: a coefficient in physical
units, and an interval around it. Everything needed comes from Part A —

| from Part A | used here |
|---|---|
| A.1 a model is a design matrix | B.2 |
| A.4 the standard error is the inverse curvature | B.2 |
| A.5 confidence interval versus prediction interval | B.3 |
| A.3 the likelihood assumes **independent** errors | B.4 |

**On scale.** Five hundred and twenty-eight months is the entire record; there is no larger
version of this analysis. Two limits are structural, and § 2.6 of the project notebook sets them
out: the forcing is reported **field-wide and monthly**, so nothing about an individual well, or
about any process faster than a month, can be asked of these data.


### B.1 · Two series on one grid

Monthly water injection and steam production for the whole field, from CalGEM, against monthly
counts of catalogue earthquakes in the field window. The magnitude threshold is M ≥ 1.2, which
§ 9.5 of the project notebook puts above the completeness magnitude for most of the record — a
count of *detected* events would otherwise track the growth of the network rather than the Earth.


In [ ]:
cat = catalog()                       # placeholder magnitudes already removed
field = cat[cat.longitude.between(FIELD[0], FIELD[1])
            & cat.latitude.between(FIELD[2], FIELD[3])].sort_values("time")
prod = production()                   # monthly field totals, already in megatonnes

edges = pd.date_range(prod.index[0], prod.index[-1] + pd.offsets.MonthBegin(1), freq="MS")
d = prod[["production", "injection"]].copy()
d["n"] = np.histogram(field[field.mag >= 1.2].time.dt.tz_localize(None), bins=edges)[0]
d = d.loc["1981":"2024"].dropna()
d = d[d.injection > 0]

print(f"{len(d)} months, {d.index[0]:%Y-%m} to {d.index[-1]:%Y-%m}")
print(f"earthquakes M >= 1.2 per month: median {d.n.median():.0f}, max {d.n.max():.0f}")


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(7.4, 5.0), sharex=True)
axs[0].plot(d.index, d.injection, lw=0.8, color="C0")
axs[0].set_ylabel("injection\n(Mt/month)", fontsize=8)
axs[1].plot(d.index, d.production, lw=0.8, color="C2")
axs[1].set_ylabel("production\n(Mt/month)", fontsize=8)
axs[2].plot(d.index, d.n, lw=0.8, color="0.3")
axs[2].set_ylabel("earthquakes\nM >= 1.2", fontsize=8)
axs[2].set_xlabel("year")
for a_ in axs:
    a_.margins(x=0.01)
axs[0].set_title("the two forcings, and the response")
fig.tight_layout(); plt.show()


### B.2 · The same regression, twice

One predictor, one response, and A.1's design matrix: $n = c_0 + c_1 x$, where $x$ is the month's
injection or its production, in megatonnes. Fit it both ways and compare — not the two
coefficients against each other, which are in different units and not comparable, but each
against **its own standard error**, which is what A.4 provides.


In [ ]:
def fit(x, y):
    """Least squares with the A.4 covariance. Returns slope, its standard error, and R^2."""
    X = np.column_stack([np.ones(len(x)), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    r = y - X @ b
    C = (r @ r / (len(y) - 2)) * np.linalg.inv(X.T @ X)
    return b, np.sqrt(np.diag(C)), 1 - (r @ r) / ((y - y.mean()) ** 2).sum(), C


y_b = d.n.values.astype(float)
ann = d.resample("YS").sum()          # section 10 correlates annual sums; both are shown

print(f"{'':>12s} {'slope (eq per Mt)':>20s} {'std errors':>12s} {'r monthly':>11s} {'r annual':>10s}")
res_b = {}
for name in ("injection", "production"):
    b, se, r2, C = fit(d[name].values, y_b)
    res_b[name] = (b, se, r2, C)
    print(f"{name:>12s} {b[1]:11.2f} +/-{se[1]:5.2f} {b[1] / se[1]:12.1f} "
          f"{d[name].corr(d.n):11.3f} {ann[name].corr(ann.n):10.3f}")


**Seismicity follows the water, and does not follow the steam.** The injection slope is thirteen
standard errors from zero; the production slope is a fifth of one, which is what a coefficient
looks like when there is no effect.

The comparison carries more weight than either fit alone. Both series are monthly field totals
over the same months, both are large industrial quantities, and both were fitted with the same
estimator; the only difference is which one is the predictor. A confound acting on the seismicity
through time — a growing network, a drifting catalogue — would raise both slopes, since production
also trends over these decades. It raises one.

A regression is also worth running where section 10 ran a correlation. The two agree
about the ordering — the annual correlations printed above are the ones that section reports —
but a correlation is a unitless number between zero and one, while the slope says how many
earthquakes accompany a megatonne of water. That is a quantity a reservoir engineer can argue
with, and it is the one that comes with an interval.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8.6, 3.6), sharey=True)
for ax, name in zip(axs, ("injection", "production")):
    b, se, r2, C = res_b[name]
    x = d[name].values
    ax.plot(x, y_b, "o", ms=2.6, mfc="0.75", mec="none", alpha=0.7)
    xs = np.linspace(x.min(), x.max(), 100)
    Xs = np.column_stack([np.ones_like(xs), xs])
    mu = Xs @ b
    vm = np.einsum("ij,jk,ik->i", Xs, C, Xs)
    ax.fill_between(xs, mu - 1.96 * np.sqrt(vm), mu + 1.96 * np.sqrt(vm), color="C0", alpha=0.5)
    ax.plot(xs, mu, "k-", lw=1.3)
    ax.set_xlabel(f"{name} (Mt/month)")
    ax.set_title(f"{name}: {b[1]:+.1f} +/- {se[1]:.1f} eq/Mt", fontsize=9)
axs[0].set_ylabel("earthquakes per month, M >= 1.2")
fig.tight_layout(); plt.show()


### B.3 · What the interval covers

A.5 distinguished two questions, and both have an answer here.

**Where is the line?** The band drawn above is the confidence interval on the fitted mean, and at
528 months it is narrow — the *average* number of earthquakes accompanying a given injection rate
is well determined.

**How many earthquakes next month?** That is the prediction interval, and it carries the scatter
of a single month as well. It does not shrink with more data, and it is the interval anyone
forecasting from this relation actually needs.


In [ ]:
b, se, r2, C = res_b["injection"]
x0 = np.array([1.0, 4.0])                       # a month injecting 4 Mt
r_i = y_b - np.column_stack([np.ones(len(d)), d.injection.values]) @ b
sigma_i = np.sqrt(r_i @ r_i / (len(d) - 2))
v_mean = x0 @ C @ x0

print(f"at 4 Mt injected in a month, the fit expects {x0 @ b:.0f} earthquakes")
print(f"  95% confidence interval on that average   +/- {1.96 * np.sqrt(v_mean):5.1f}")
print(f"  95% prediction interval for one month     +/- {1.96 * np.sqrt(v_mean + sigma_i**2):5.1f}")
print(f"  ratio {np.sqrt((v_mean + sigma_i**2) / v_mean):.0f}x")
print(f"\nR^2 = {r2:.3f}: injection accounts for about a quarter of the month-to-month variance,")
print("so three quarters of what happens in any given month is something else.")


### B.4 · One reason to distrust the error bar

Thirteen standard errors is a large number, and it rests on an assumption A.3 stated in its first
line: that the errors are **independent**. Independence is what turned a product of densities into
a sum, and A.4's covariance inherits it entirely.

Earthquakes at The Geysers come in swarms lasting weeks to months, so a month with more events
than the model predicts is likely to be followed by another one. That is a claim about the
residuals, and it takes one line to check.


In [ ]:
r_full = y_b - np.column_stack([np.ones(len(d)), d.injection.values]) @ b
ac1 = np.corrcoef(r_full[:-1], r_full[1:])[0, 1]
n_eff = len(d) * (1 - ac1) / (1 + ac1)
print(f"lag-1 autocorrelation of the residuals: {ac1:+.3f}")
print(f"  independent months would give roughly {1.96 / np.sqrt(len(d)):+.3f}")
print(f"  so {len(d)} months carry about as much information as {n_eff:.0f} independent ones,")
print(f"  and the standard error is understated by roughly {np.sqrt(len(d) / n_eff):.1f}x")


So the honest figure is not thirteen standard errors but something nearer six. The conclusion does
not change — the injection slope was never close to zero, and the production slope was never far
from it — but the *precision* did, by a factor of two, and nothing in the output of B.2 hinted at
it. The estimate stays; the interval around it was computed under an assumption the data do not
satisfy.

That is the same lesson Part C reaches from the other direction, and the exercise below prices it.


**Exercise 4.** The standard error above assumes each month is an independent draw. A **moving-
block bootstrap** does not: it resamples contiguous runs of months, so a swarm is carried along
intact.

Resample the months in blocks of 1, 6, 12, 24 and 36 months, refit the injection regression on
each resample, and take the spread of the slope over 400 resamples. Predict first what the
one-month block must give, and why.


In [ ]:
# your code here


The one-month block reproduces the formula, which is the check that the machinery is right:
resampling months independently *is* the assumption the formula encodes. Lengthening the block
until it spans a swarm roughly doubles the interval, in line with the autocorrelation estimate
above, and then it stops growing — which is what a correlation that dies away within a year
should do.


## Part C · The b-value, and what its error bar leaves out

The b-value is the slope of the frequency–magnitude distribution: how many small earthquakes there
are for each large one. It enters hazard estimates and every claim that seismicity has changed
character.

**Section 9 of the project notebook has already done the estimation**, and this part does not repeat
it. It establishes there that the b-value is estimated by maximum likelihood rather than by fitting
a line to the cumulative distribution, that maximum curvature and goodness of fit are two ways to
choose the completeness magnitude, and — § 9.4 — that Utsu's binning correction must not be applied
to magnitudes that were never binned.

Section 9 also states a caution it does not quantify:

> *The standard error b/√n assumes independent draws from an exponential distribution. Aftershocks
> and swarms are not independent, and the seismicity of this field is almost entirely swarms, so the
> true interval is wider than the formula gives. The quantity should be treated as a lower bound.*

Part B has just measured exactly that kind of gap for the injection regression. This part measures it
for the b-value, using A.3 to see where the assumption enters and A.5 to price it.


### C.1 · Where independence enters

A.3's recipe, applied to magnitudes. Above a completeness magnitude $M_c$ the Gutenberg–Richter law
$\log_{10}N(\ge M) = a - bM$ makes the survival function $P(M' > M) = e^{-\beta(M-M_c)}$ with
$\beta = b\ln 10$, so a magnitude is an **exponential** random variable.

1. **The likelihood.** For $n$ magnitudes,
   $L(\beta) = \prod_i \beta e^{-\beta(M_i - M_c)}$ — and that product is a product **only because
   the events are assumed independent**. This is the single step where the assumption enters, and
   everything below inherits it.
2. **Maximise.** $\ell(\beta) = n\log\beta - \beta\sum_i(M_i - M_c)$, so $\hat\beta = 1/(\bar M -
   M_c)$ and $\hat b = \log_{10}e/(\bar M - M_c)$ — Aki's estimator, needing only the mean magnitude.
3. **The curvature.** $-\partial^2\ell/\partial\beta^2 = n/\beta^2$, so by A.4
   $\sigma_b = b/\sqrt{n}$.

The $n$ in that last formula counts **events**. If the events are not independent draws, $n$ is not
the number of independent pieces of information, and the formula is answering a question nobody
asked.


In [ ]:
# ── Checkpoint 5 ── run this if you are behind or something broke ──
# Part C needs only the field-window catalogue that Part B built. Loading it here costs
# nothing when Part B has already run, and lets this part stand on its own otherwise.
if "field" not in globals():
    from geysers_data import catalog, FIELD
    cat = catalog()
    field = cat[cat.longitude.between(FIELD[0], FIELD[1])
                & cat.latitude.between(FIELD[2], FIELD[3])].sort_values("time")
print(f"{len(field):,} events in the field window")


In [ ]:
MC = 1.2
mags = field[field.mag >= MC].mag.values
print(f"{len(mags):,} events at or above M {MC} in the field window")
print(f"distinct magnitude values: {pd.Series(mags).round(2).nunique()};  "
      f"on a 0.1 grid: {100 * np.mean(np.abs(mags * 10 - np.round(mags * 10)) < 1e-6):.1f} %")
print("  -> reported to 0.01, so no binning correction is applied (project notebook, section 9.4)")

b_c = np.log10(np.e) / (mags.mean() - MC)
sb_c = b_c / np.sqrt(len(mags))
print(f"\nb = {b_c:.4f}")
print(f"sigma_b = b/sqrt(n) = {sb_c:.4f}   -- a 95% interval of +/-{1.96 * sb_c:.4f}")


### C.2 · Pricing the dependence

Two resamplings, exactly as in B.5. Drawing **events** with replacement assumes they are
independent, and must reproduce the formula. Drawing **contiguous intervals of time** keeps each
swarm intact, so whatever the swarms do to the estimate survives into the spread.


**Exercise 5.** Estimate the uncertainty on $b$ two ways. First resample the magnitudes
themselves with replacement; then resample contiguous blocks of time — 1, 7, 30, 180 and 365 days —
keeping every event inside a chosen block.

One of these must agree with $b/\sqrt{n}$ by construction. Say which, and why, before running it.
Then read the trend across block lengths as a measurement of how far the events depart from
independence.


In [ ]:
# your code here


The event-level bootstrap lands on the formula, as it must: resampling events independently
is the assumption the formula encodes, so the two are the same calculation by different means.
Neither is evidence that the assumption is true.

Resampling time is a different question, and the answer keeps growing across every block length
tried, without settling. Had the only dependence been swarms lasting weeks, the curve would have
flattened once a block reliably contained one; it does not, which says the catalogue carries
structure on longer timescales too — the field's operations changed over four decades, and so did
the network that recorded it.

So the honest interval is not a single number. What can be said is its direction and the least of
it: several times the textbook value, and still rising where the measurement runs out. Section 9.4
called b/√n a lower bound without saying how loose; it is now measured, and it is loose by at
least a factor of several.

The longest blocks carry their own caveat, visible in the last points of the figure: a two-year
block leaves only a couple of dozen blocks to resample, so the estimate of the spread is itself
becoming noisy there.


In [ ]:
# ── Checkpoint 6 ── run this if you are behind or something broke ──
# The figure below needs what Exercise 5 built, so it is rebuilt here and a wrong answer
# above cannot strand the rest of the section.
rng_c = np.random.default_rng(207)
sub = field[field.mag >= MC][["time", "mag"]].reset_index(drop=True)
day = (sub.time.dt.floor("D") - sub.time.dt.floor("D").min()).dt.days


def b_of(m):
    return np.log10(np.e) / (m.mean() - MC)


print(f"{len(sub):,} magnitudes above M {MC}, spanning {day.max():,} days")


In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.6))
spans = np.array([1, 3, 7, 14, 30, 90, 180, 365, 730])
vals = []
for span in spans:
    blocks = [g.values for _, g in sub.groupby(day // span).mag]
    reps = [b_of(np.concatenate([blocks[i] for i in
                                 rng_c.integers(0, len(blocks), len(blocks))]))
            for _ in range(150)]
    vals.append(np.std(reps))
vals = np.array(vals)
ax.semilogx(spans, vals / sb_c, "o-", ms=4, color="C0")
ax.axhline(1.0, color="C3", ls="--", lw=0.9)
ax.text(1.2, 1.05, r"the formula, $b/\sqrt{n}$", color="C3", fontsize=7.5)
ax.set_xlabel("bootstrap block length (days)")
ax.set_ylabel(r"interval width / $\sigma_b$")
ax.set_title("how much of the error bar independence was buying")
fig.tight_layout(); plt.show()


### C.3 · What this means for a reported b-value

$\sigma_b = b/\sqrt{n}$ is not wrong. It is exactly right about the thing it describes — how much
$\hat b$ would vary if these magnitudes were redrawn independently from one exponential
distribution. The catalogue was not produced that way, and the difference is a factor of several.

That matters because b-values are compared. A change of a few hundredths between two periods, or
between two parts of a field, is a routine claim in the literature, and it is often supported by
intervals computed from $b/\sqrt{n}$ on tens of thousands of events. Those intervals are narrower
than the data support, by roughly the factor measured above, before any of the other choices — the
completeness cut, the window, the treatment of the magnitude mixture — are considered at all.

Section 9.4 names the last of those: the magnitudes here are a mixture of coda duration below about
magnitude 3 and local and moment magnitude above it, so a slope fitted across the range crosses a
change of scale. None of that appears in $\sigma_b$ either.


## Takeaways

### Method

- A relation that is linear in its *parameters* is a design matrix, whatever non-linear functions
  of the measurements its columns contain. Choosing those columns is the science; the solve is the
  same every time.
- Least squares is not a definition. It is maximum likelihood for one noise model, and the object
  worth carrying is the recipe underneath: write the likelihood, maximise it, invert its curvature.
  That recipe produced the normal equations in Part A and Aki's b-value in Part C, which share no
  algebra at all.
- The error bar on any estimate is the inverse curvature of the log-likelihood at its maximum.
- Running the same regression against a second predictor is a stronger argument than either fit
  alone. A confound acting through time would have lifted both slopes; it lifted one.

### Uncertainty

- The interval on *where the line is* shrinks as more data arrive. The interval on *what the next
  measurement will be* does not, and it is the one a forecast needs.
- Correlated columns leave the combination determined and the individual coefficients undetermined,
  so a coefficient can disagree with a published value by many standard errors while both curves
  pass through the same data.
- Ridge regression is the maximum-a-posteriori estimate under a Gaussian prior, with a penalty that
  is the noise variance over the prior variance. It buys something when data are scarce and nothing
  when they are plentiful, and it is what makes a rank-deficient design solvable at all.
- **An interval describes one source of variation and is silent about every other.** Both
  applications showed this: monthly seismicity is autocorrelated, and catalogue magnitudes arrive
  in swarms, so in each case the standard error computed under an independence assumption was
  optimistic by a factor of two or more. Nothing in the output that produced it indicated this.

### Seismology

- Seismicity at The Geysers follows water injection and does not follow steam production. The
  injection slope is thirteen standard errors from zero and the production slope is a fifth of one,
  from the same estimator on the same months — which is a cleaner argument than either number
  alone. Eberhart-Phillips and Oppenheimer (1984) found no consistent correlation in the first
  decade of this record, and the project notebook explains why that was a statement about the
  catalogue rather than about the Earth.
- Injection accounts for about a quarter of the month-to-month variance in seismicity rate, so
  anything forecast from it carries a prediction interval far wider than the confidence interval
  on its slope. Both were computed here, and they differ by more than an order of magnitude.
- Above the completeness magnitude, earthquake magnitudes are exponentially distributed, so the
  b-value has a closed-form maximum-likelihood estimator needing only the mean magnitude
  (Aki 1965; the binning correction is Utsu's, and applies only where magnitudes were binned).
- The magnitude column of this catalogue is a mixture of scales, overwhelmingly coda duration, so a
  b-value fitted across its full range is fitted across a change of scale — as the project
  notebook's magnitude section sets out.